In [13]:
import sys
import os
import pandas as pd
import numpy as np

In [14]:
sys.path.append(os.path.abspath("C:/01_Data/PythonProgram/Toxic_Comment_Classification_PROJECT/src"))
from text_preprocessing_gru import clean_text_dl

In [15]:
df = pd.read_csv(
    "C:/01_Data/PythonProgram/Toxic_Comment_Classification_PROJECT/train_data/processed_train_model2.csv")
df["clean_text"] = df["comment_text"].apply(clean_text_dl)

In [16]:
from sklearn.model_selection import train_test_split
x=df['clean_text'].values
y=df['is_toxic'].values

In [17]:
#using stratified train test split because of the imbalanced data
x_train, x_val, y_train, y_val = train_test_split(x, y,test_size=0.2,random_state=42,stratify=y)

In [18]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer=Tokenizer(num_words=50000,oov_token='unknown')
tokenizer.fit_on_texts(x_train)
print(len(tokenizer.word_index))

175638


In [19]:
train_seq = tokenizer.texts_to_sequences(x_train)
val_seq = tokenizer.texts_to_sequences(x_val)

In [20]:
x_train_pad = pad_sequences(train_seq, maxlen=200, padding="post", truncating="post")
x_val_pad = pad_sequences(val_seq, maxlen=200, padding="post", truncating="post")

In [21]:
x_train_pad.shape

(127656, 200)

In [22]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1])
class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print(class_weight_dict)

{0: np.float64(0.5565942307021522), 1: np.float64(4.917411402157165)}


In [23]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, GRU, Dense, Dropout

EMBED_DIM = 300

model = Sequential([
    Embedding(input_dim=50000, output_dim=EMBED_DIM, input_length=200),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.4),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

model.compile(loss="binary_crossentropy",optimizer='adam',metrics=["accuracy"])

C:\Anaconda\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [24]:
model = Sequential([
    Embedding(input_dim=50000, output_dim=EMBED_DIM, input_length=200),
    Bidirectional(GRU(64, return_sequences=False)),
    Dropout(0.4),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

model.compile(loss="binary_crossentropy",optimizer='adam',metrics=["accuracy"])

In [25]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=0, restore_best_weights=True)
]

history = model.fit(x_train_pad, y_train, validation_data=(x_val_pad, y_val),epochs=5, batch_size=128, class_weight=class_weight_dict, callbacks=callbacks)

Epoch 1/5
998/998 ━━━━━━━━━━━━━━━━━━━━ 1573s 2s/step - accuracy: 0.9011 - loss: 0.2660 - val_accuracy: 0.8986 - val_loss: 0.2598
Epoch 2/5
998/998 ━━━━━━━━━━━━━━━━━━━━ 5103s 5s/step - accuracy: 0.9445 - loss: 0.1367 - val_accuracy: 0.9390 - val_loss: 0.1440
Epoch 3/5
998/998 ━━━━━━━━━━━━━━━━━━━━ 9811s 10s/step - accuracy: 0.9641 - loss: 0.0837 - val_accuracy: 0.9351 - val_loss: 0.1733


In [26]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score,classification_report,roc_auc_score

thresholds = np.arange(0.05, 0.96, 0.01)

best_t = 0
best_f1 = 0

y_val_probs = model.predict(x_val_pad).ravel()

for t in thresholds:
    y_pred = (y_val_probs >= t).astype(int)
    f1 = f1_score(y_val, y_pred)

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best Threshold:", best_t)

y_best_pred = (y_val_probs >= best_t).astype(int)

print("Precision:", precision_score(y_val, y_best_pred))
print("Recall:", recall_score(y_val, y_best_pred))
print("F1:", f1_score(y_val, y_best_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_val_probs))
print(classification_report(y_val, y_best_pred))


998/998 ━━━━━━━━━━━━━━━━━━━━ 62s 61ms/step 
Best Threshold: 0.8300000000000002
Precision: 0.853516295025729
Recall: 0.7667180277349769
F1: 0.8077922077922078
ROC-AUC: 0.9763132623880585
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     28670
           1       0.85      0.77      0.81      3245

    accuracy                           0.96     31915
   macro avg       0.91      0.88      0.89     31915
weighted avg       0.96      0.96      0.96     31915



In [27]:
def predict_toxic(text):
    text = clean_text_dl(text)
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=200, padding="post", truncating="post")
    prob = model.predict(pad)[0][0]
    return prob

print(predict_toxic("you are a nice person"))
print(predict_toxic("you are a stupid idiot"))
print(predict_toxic("go kill yourself"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step
0.75575274
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
0.99958307
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
0.9979217
